# Featuring Engineering - The iPinYou Dataset

In [2]:
# Data manipulation
import numpy as np
import pandas as pd

# Visualisation (remove if unused)
import matplotlib.pyplot as plt
import seaborn as sns

# Encoding / preprocessing
from sklearn import set_config
from sklearn.preprocessing import LabelEncoder, TargetEncoder

# Model selection
from sklearn.model_selection import KFold

# Machine learning models
from sklearn.ensemble import (
    RandomForestClassifier,   # treatment (binary)
    RandomForestRegressor     # outcome (continuous)
)

# Statistical modelling
import statsmodels.api as sm

from sklearn.preprocessing import OneHotEncoder

from sklearn.preprocessing import MultiLabelBinarizer


# Return transformed outputs as DataFrames
set_config(transform_output="pandas")

#### Loading the Data 

In [13]:
# Load advertiser 1458 training data from local machine
from ad_causal_doubleml.config.paths import DATA_DIR
file_path = DATA_DIR / "train.log.txt"

In [16]:
#2 min runtime

# explicit dtypes reduce memory usage when loading
# the large dataset
dtypes = {
    "click": "int8",
    "weekday": "int8",
    "hour": "int8",
    "logtype": "int8",

    "region": "int16",
    "city": "int16",

    "adexchange": "category",

    "slotwidth": "int16",
    "slotheight": "int16",

    "slotvisibility": "category",
    "slotformat": "int16",

    "slotprice": "int16",

    "bidprice": "int16",
    "payprice": "int16",

    "advertiser": "category",

    # High-cardinality categorical features
    "bidid": "string",
    "timestamp": "string",
    "ipinyouid": "string",
    "useragent": "category",
    "IP": "string",
    "domain": "string",
    "url": "string",
    "urlid": "string",
    "slotid": "string",
    "creative": "string",
    "keypage": "string",
    "usertag": "string"
}

df = pd.read_csv(
    file_path,
    sep="\t",
    dtype=dtypes,
    na_values=["null"]
)

In [18]:
df['slotformat'].value_counts()

slotformat
0    8358845
1    3795822
5      82420
Name: count, dtype: int64

In [ ]:
df["slotformat"].value_counts().sort_index()

slotformat
0    8358845
1    3795822
5      82420
Name: count, dtype: int64

In [25]:
# investigate unique variables to see which to one-hot encode and which to target encode

print(df.nunique())

# User Tags 

# user tags, for example '10052,10006,10110'
# Count unique tags in usertag column to estimate dimensionality

unique_tags = (
    df['usertag']
    .dropna()                 # remove missing values
    .astype(str)
    .str.split(',')           # split comma-separated tags
    .explode()                # one tag per row
    .str.strip()              # remove spaces if any
    .nunique()                # count unique tags
)

print(f"Number of unique user tags: {unique_tags}")

TypeError: unhashable type: 'list'

### Columns to Drop 

In [ ]:
df = df.drop(columns=['bidid','logtype','ipinyouid','IP','url','urlid','payprice'])

### Columns to One-Hot Encode 

In [ ]:
variables_one_hot = ['adexchange', 'useragent','weekday','region','slotwidth','slotheight','slotvisibility','slotformat','bidprice', 'advertiser','keypage']

### Multi Label Columns to One-Hot Encode

In [24]:
variables_multi_label_one_hot_encode = 'usertag'

### Columns to Target Encode

In [ ]:
target_categorical_features = ['city','domain','slotid','slotprice','creative']

### Cyclical Encoding 

In [ ]:
df['hour_sin'] = np.sin(2*np.pi*df['hour']/24)
df['hour_cos'] = np.cos(2*np.pi*df['hour']/24)
df = df.drop(columns=['hour'])

### One Hot Encoder

In [ ]:
# data for one-hot-enoder
sample_df_one_hot = df[variables_one_hot]
encoder = OneHotEncoder(sparse_output=False)


# one-hot encode the low dimensionality features
# this creates a new column for each of the categories within a feature
enc = encoder.fit_transform(sample_df_one_hot)

# converting arrays to a df
encoded_colm = pd.DataFrame(enc,
                            columns=encoder.get_feature_names_out(sample_df_one_hot.columns),
                            index=sample_df.index)
# (run time 8s with sample of 100,000)

sample_df_with_one_hot = pd.concat([sample_df, encoded_colm], axis=1)

### MultiLabelBinarizer: One-Hot Encoder

In [ ]:
# dealing with user tags seperately due to multiple categories being in one columns
# output is one-hot encoding 

df['usertag_list'] = (
    df['usertag']
    .fillna('')
    .str.split(',')
)

mlb = MultiLabelBinarizer()

tag_matrix = mlb.fit_transform(df['usertag_list'])

tag_df = pd.DataFrame(
    tag_matrix,
    columns=[f'usertag_{t}' for t in mlb.classes_],
    index=df.index
)

df = pd.concat(
    [df.drop(columns=['usertag', 'usertag_list']),
     tag_df],
    axis=1
)

### Dataset Ready for export

In [1]:
df['slotformat']

NameError: name 'df' is not defined